# Colab Compatibility Verification

Verifies fixes for Colab-specific issues found in the audit.
Run **all cells in order** on Google Colab. Each section has a ✅/❌ checklist.

## Issues Covered

| ID | Issue | Priority |
|----|-------|----------|
| A-1 | Consecutive jobs (Tune→Fit, Fit→Fit) — polling restart | P0 |
| B-1 | Plot tab rapid switching — request_id stale filter | P1 |
| B-2 | Inference button double-click guard | P2 |
| B-3 | YAML Export button double-click guard | P2 |
| B-5 | Config form disabled during job | P3 |
| C-1 | _run_job TOCTOU atomic guard | P1 |
| C-2 | _model thread-safe access | P1 |

## 0. Install (Colab only)

In [ ]:
# For testing from a branch (replace with PyPI install for released version):
# !pip install -q "lizyml-widget @ git+https://github.com/user/LizyML-Widget.git@fix/colab-action-dispatch"
!pip install -q lizyml-widget lizyml[plots,tuning,calibration,explain]

## 1. Environment Check

In [ ]:
import sys
print(f"Python: {sys.version}")

import anywidget, ipywidgets, traitlets, lizyml_widget
print(f"anywidget:      {anywidget.__version__}")
print(f"ipywidgets:     {ipywidgets.__version__}")
print(f"traitlets:      {traitlets.__version__}")
print(f"lizyml_widget:  {lizyml_widget.__version__}")

try:
    import google.colab
    print("Environment:    Google Colab ✅")
except ImportError:
    print("Environment:    NOT Colab (results may differ)")

# Verify thread safety attributes exist
from lizyml_widget import LizyWidget
import threading
w_check = LizyWidget()
assert hasattr(w_check, "_job_lock"), "_job_lock missing (C-1 not applied)"
assert isinstance(w_check._job_lock, type(threading.Lock())), "_job_lock wrong type"
assert hasattr(w_check._service, "_model_lock"), "_model_lock missing (C-2 not applied)"
print("Thread safety:  _job_lock ✅, _model_lock ✅")
del w_check

## 2. Load Widget + Data

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from lizyml_widget import LizyWidget

data = load_breast_cancer(as_frame=True)
df = data.frame
print(f"Shape: {df.shape}, Target: 'target' ({df['target'].nunique()} classes)")

w = LizyWidget()
w.load(df, target="target")
print(f"Status: {w.status}")
w

## 3. Scenario A-1: Consecutive Jobs (P0 — Critical)

**The main Colab bug.** Tests that Fit→Fit and Tune→Apply→Fit work.

### 3a. First Fit (via Python API)

In [ ]:
# First Fit — should complete normally
w.fit()
# Wait for job to finish
if w._job_thread:
    w._job_thread.join(timeout=120)

print(f"Job 1 — status: {w.status}, job_index: {w.job_index}")
print(f"  fit_summary keys: {list(w.fit_summary.keys()) if w.fit_summary else 'EMPTY'}")
assert w.status == "completed", f"Expected completed, got {w.status}: {w.error}"
assert w.job_index == 1, f"Expected job_index=1, got {w.job_index}"
print("✅ First Fit completed")

### 3b. Second Fit (Fit→Fit)

**This is the A-1 bug scenario.** On pre-fix Colab, the UI would not respond to the second job.

In [ ]:
# Second Fit — this is the A-1 regression test
prev_index = w.job_index
w.fit()
if w._job_thread:
    w._job_thread.join(timeout=120)

print(f"Job 2 — status: {w.status}, job_index: {w.job_index}")
assert w.status == "completed", f"Expected completed, got {w.status}: {w.error}"
assert w.job_index == prev_index + 1, f"job_index should increment: {prev_index} -> {w.job_index}"
print("✅ Second Fit (Fit→Fit) completed — A-1 polling restart works")

### 3c. Tune → Apply Best Params → Fit

The original user-reported scenario.

In [ ]:
# Tune
prev_index = w.job_index
w.tune()
if w._job_thread:
    w._job_thread.join(timeout=300)

print(f"Tune — status: {w.status}, job_index: {w.job_index}")
assert w.status == "completed", f"Tune failed: {w.error}"
tune_best = w.tune_summary.get("best_params", {})
print(f"  best_params: {tune_best}")
assert tune_best, "No best_params from tune"
print("✅ Tune completed")

In [ ]:
# Apply best params → Fit
w._handle_apply_best_params({"params": tune_best})
print(f"Config after apply: lr={w.config.get('model', {}).get('params', {}).get('learning_rate')}")

prev_index = w.job_index
w.fit()
if w._job_thread:
    w._job_thread.join(timeout=120)

print(f"Fit after Tune — status: {w.status}, job_index: {w.job_index}")
assert w.status == "completed", f"Fit after Tune failed: {w.error}"
assert w.job_index == prev_index + 1
print("✅ Tune → Apply → Fit completed — A-1 scenario verified")

## 4. Scenario C-1: Double-Click Guard (TOCTOU)

Rapid double invocation of fit must start only one job.

In [ ]:
import threading

prev_index = w.job_index

# Simulate rapid double-click by calling fit from two threads
t1 = threading.Thread(target=w.fit)
t2 = threading.Thread(target=w.fit)
t1.start()
t2.start()
t1.join(timeout=120)
t2.join(timeout=120)
if w._job_thread:
    w._job_thread.join(timeout=120)

print(f"After double-call — job_index: {w.job_index} (prev: {prev_index})")
assert w.job_index == prev_index + 1, f"Expected exactly 1 new job, got {w.job_index - prev_index}"
assert w.status == "completed", f"Expected completed, got {w.status}: {w.error}"
print("✅ C-1: Double-click guard works — only 1 job started")

## 5. Scenario B-1: Plot Request ID Echo

Verifies that plot responses carry `request_id` for stale-response filtering.

In [ ]:
from unittest.mock import MagicMock

sent = []
w.send = MagicMock(side_effect=lambda msg, **kw: sent.append(msg))

# Request with request_id
w._handle_request_plot({"plot_type": "learning_curve", "request_id": "test-42"})
assert len(sent) >= 1, "No response sent"
resp = sent[-1]
print(f"Response type: {resp['type']}, plot_type: {resp.get('plot_type')}")
if resp["type"] == "plot_data":
    assert resp.get("request_id") == "test-42", f"request_id mismatch: {resp.get('request_id')}"
    print("✅ B-1: request_id echoed in plot_data")
elif resp["type"] == "plot_error":
    assert resp.get("request_id") == "test-42", f"request_id mismatch: {resp.get('request_id')}"
    print("✅ B-1: request_id echoed in plot_error")

# Request without request_id (backward compat)
sent.clear()
w._handle_request_plot({"plot_type": "feature_importance"})
resp2 = sent[-1]
assert resp2.get("request_id") is None, f"Unexpected request_id: {resp2.get('request_id')}"
print("✅ B-1: No request_id when not provided (backward compat)")

# Restore send
del w.send

## 6. Scenario C-2: Model Lock Thread Safety

Verifies that `_model_lock` protects model access during concurrent operations.

In [ ]:
import time

lock = w._service._model_lock
blocked = True

def try_get_model():
    global blocked
    # This should block until lock is released
    m = w._service.get_model()
    blocked = False

# Hold the lock, attempt get_model from another thread
with lock:
    t = threading.Thread(target=try_get_model)
    t.start()
    time.sleep(0.3)
    assert blocked, "get_model() should block while lock is held"
    print("✅ get_model() correctly blocks while _model_lock is held")

t.join(timeout=5)
assert not blocked, "get_model() should have completed after lock release"
print("✅ C-2: _model_lock thread safety verified")

## 7. Scenario B-2, B-3, B-5: UI Guards (Manual)

These must be verified **visually in the widget UI** above (Section 2).

### B-2: Inference Button Guard
1. Go to **Results** tab → Inference section
2. Load inference data: run the cell below, then click **Run Inference**
3. **Verify**: Button shows "Running..." and is disabled while processing
4. Click rapidly — only one request should execute

### B-3: YAML Export Guard
1. Go to **Model** tab → scroll to footer buttons
2. Click **Export YAML** rapidly
3. **Verify**: Button shows "Exporting..." and is disabled, only 1 file downloaded

### B-5: Config Form Disabled During Job
1. Go to **Model** tab
2. Click **Fit** (from UI, not Python)
3. **While running**: Try to change any config slider/dropdown
4. **Verify**: Form is grayed out (`opacity: 0.6`) and unclickable

In [ ]:
# Load inference data for B-2 test
df_test = df.drop(columns=["target"]).tail(20)
w.load_inference(df_test)
print(f"Inference data loaded: {len(df_test)} rows")
print("Now go to Results tab → Inference section and click 'Run Inference'")
print("Verify: button shows 'Running...' and is disabled during processing")

## 8. Full UI Scenario (Manual — the most important test)

**Do all of the following using the widget UI only** (no Python cells):

1. **Data tab**: Select a different target → select back to `target`
2. **Model tab (Fit)**: Change `n_estimators` slider → click **Fit**
3. **While running**: Observe progress bar updating (polling works)
4. **After completion**: Switch to **Results** tab → see metrics
5. **Results tab**: Click through plot tabs rapidly (ROC, Feature Importance, etc.)
6. **Model tab (Tune)**: Click **Tune** → wait for completion
7. **Results tab**: Click **Apply to Fit ▸** button
8. **Model tab**: Verify config updated → click **Fit**
9. **After completion**: Verify new results appear (not stale Tune results)
10. **Results tab**: Load inference data (cell above) → click **Run Inference**
11. **Model tab footer**: Click **Export YAML** → verify download

### Checklist

| Step | Expected | ✅/❌ |
|------|----------|------|
| 1. Target select | df_info updates, columns refresh | |
| 2. Fit starts | Progress bar appears, status="running" | |
| 3. Progress updates | Elapsed time ticks, fold progress shows | |
| 4. Fit completes | Metrics table shows, plots available | |
| 5. Plot switching | Correct plots, no flicker/corruption | |
| 6. Tune completes | Best params shown, trials listed | |
| 7. Apply to Fit | Switches to Model tab, params applied | |
| 8. Fit after Tune | **NEW** fit starts (not stuck on old results) | |
| 9. New results | fit_summary reflects new params | |
| 10. Inference | Prediction table appears | |
| 11. YAML Export | config.yaml downloads | |

## 9. State Dump (run after manual test)

Run this cell after completing the manual UI test to capture final state.

In [ ]:
print("=== Final Widget State ===")
print(f"status:         {w.status}")
print(f"job_type:       {w.job_type}")
print(f"job_index:      {w.job_index}")
print(f"error:          {dict(w.error) if w.error else '{}'}")
print(f"fit_summary:    {bool(w.fit_summary)} (keys: {list(w.fit_summary.keys()) if w.fit_summary else []})")
print(f"tune_summary:   {bool(w.tune_summary)} (keys: {list(w.tune_summary.keys()) if w.tune_summary else []})")
print(f"avail_plots:    {list(w.available_plots)}")
print(f"inference:      {w.inference_result.get('status', 'N/A')}")
print(f"thread alive:   {w._job_thread.is_alive() if w._job_thread else 'N/A'}")

# Summary
issues = []
if w.status != "completed":
    issues.append(f"status={w.status} (expected completed)")
if w.job_index < 4:
    issues.append(f"job_index={w.job_index} (expected ≥4 after all scenarios)")
if not w.fit_summary:
    issues.append("fit_summary is empty")

if issues:
    print(f"\n❌ Issues found: {issues}")
else:
    print(f"\n✅ All automated checks passed. Total jobs run: {w.job_index}")